In [2]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

# Load preprocessed data
print("Loading preprocessed data...")
df = pd.read_pickle(r'C:\Users\varun\Desktop\fraud-gnn\data\preprocessed_data.pkl')
print(f"Shape: {df.shape}")
print("✅ Loaded!")

Loading preprocessed data...
Shape: (590540, 33)
✅ Loaded!


In [3]:
# Build the graph
# Nodes = unique cards
# Edges = transactions between cards

print("Building graph...")

# Create node mapping - each unique card1 value is a node
unique_cards = df['card1'].unique()
card_to_node = {card: idx for idx, card in enumerate(unique_cards)}
num_nodes = len(unique_cards)
print(f"Number of nodes (unique cards): {num_nodes}")

# Create edges - each transaction is an edge
print("Creating edges...")
src_nodes = df['card1'].map(card_to_node).values
dst_nodes = df['card1'].map(card_to_node).values

# Edge index (PyG format needs 2 x num_edges tensor)
edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)
print(f"Number of edges (transactions): {edge_index.shape[1]}")

# Node features - aggregate transaction features per card
print("Creating node features...")
feature_cols = ['TransactionAmt', 'C1', 'C2', 'C3', 'C4', 'C5',
                'V1', 'V2', 'V3', 'V4', 'V5']
node_features = df.groupby('card1')[feature_cols].mean()
node_features = node_features.reindex(unique_cards).fillna(0).values
x = torch.tensor(node_features, dtype=torch.float)
print(f"Node feature matrix shape: {x.shape}")

# Labels - is this card associated with fraud?
print("Creating labels...")
card_fraud = df.groupby('card1')['isFraud'].max()
card_fraud = card_fraud.reindex(unique_cards).fillna(0).values
y = torch.tensor(card_fraud, dtype=torch.long)
print(f"Labels shape: {y.shape}")
print(f"Fraud nodes: {y.sum().item()} out of {len(y)}")

print("\n✅ Graph built successfully!")

Building graph...
Number of nodes (unique cards): 13553
Creating edges...


C:\Users\varun\AppData\Local\Temp\ipykernel_12980\1594945066.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([src_nodes, dst_nodes], dtype=torch.long)


Number of edges (transactions): 590540
Creating node features...
Node feature matrix shape: torch.Size([13553, 11])
Creating labels...
Labels shape: torch.Size([13553])
Fraud nodes: 1740 out of 13553

✅ Graph built successfully!


In [4]:
# Fix the warning and create PyG Data object
edge_index = torch.tensor(
    np.array([src_nodes, dst_nodes]), 
    dtype=torch.long
)

# Create the PyTorch Geometric Data object
data = Data(x=x, edge_index=edge_index, y=y)

print("=== GRAPH SUMMARY ===")
print(f"Nodes: {data.num_nodes}")
print(f"Edges: {data.num_edges}")
print(f"Node features: {data.num_node_features}")
print(f"Has isolated nodes: {data.has_isolated_nodes()}")
print(f"Has self loops: {data.has_self_loops()}")
print(f"Is directed: {data.is_directed()}")

# Save the graph
torch.save(data, r'C:\Users\varun\Desktop\fraud-gnn\data\graph_data.pt')
print("\n✅ Graph saved!")

=== GRAPH SUMMARY ===
Nodes: 13553
Edges: 590540
Node features: 11
Has isolated nodes: True
Has self loops: True
Is directed: False

✅ Graph saved!


In [5]:
# Use a smaller sample to avoid RAM issues
print("Creating smaller graph sample...")

# Take only 50K transactions instead of 590K
df_sample = df.sample(n=50000, random_state=42)

# Rebuild graph with sample
unique_cards = df_sample['card1'].unique()
card_to_node = {card: idx for idx, card in enumerate(unique_cards)}
num_nodes = len(unique_cards)

src_nodes = df_sample['card1'].map(card_to_node).values
dst_nodes = df_sample['card1'].map(card_to_node).values

edge_index = torch.tensor(
    np.array([src_nodes, dst_nodes]),
    dtype=torch.long
)

feature_cols = ['TransactionAmt', 'C1', 'C2', 'C3', 'C4', 'C5',
                'V1', 'V2', 'V3', 'V4', 'V5']
node_features = df_sample.groupby('card1')[feature_cols].mean()
node_features = node_features.reindex(unique_cards).fillna(0).values
x = torch.tensor(node_features, dtype=torch.float)

card_fraud = df_sample.groupby('card1')['isFraud'].max()
card_fraud = card_fraud.reindex(unique_cards).fillna(0).values
y = torch.tensor(card_fraud, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)

print(f"Nodes: {data.num_nodes}")
print(f"Edges: {data.num_edges}")
print(f"Fraud nodes: {y.sum().item()}")

# Save smaller graph
torch.save(data, r'C:\Users\varun\Desktop\fraud-gnn\data\graph_data.pt')
print("✅ Smaller graph saved!")

Creating smaller graph sample...
Nodes: 5657
Edges: 50000
Fraud nodes: 555
✅ Smaller graph saved!


In [6]:
# Super small sample - just 10K transactions
df_sample = df.sample(n=10000, random_state=42)

unique_cards = df_sample['card1'].unique()
card_to_node = {card: idx for idx, card in enumerate(unique_cards)}

src_nodes = df_sample['card1'].map(card_to_node).values
dst_nodes = df_sample['card1'].map(card_to_node).values

edge_index = torch.tensor(
    np.array([src_nodes, dst_nodes]),
    dtype=torch.long
)

feature_cols = ['TransactionAmt', 'C1', 'C2', 'C3', 'C4', 'C5',
                'V1', 'V2', 'V3', 'V4', 'V5']
node_features = df_sample.groupby('card1')[feature_cols].mean()
node_features = node_features.reindex(unique_cards).fillna(0).values
x = torch.tensor(node_features, dtype=torch.float)

card_fraud = df_sample.groupby('card1')['isFraud'].max()
card_fraud = card_fraud.reindex(unique_cards).fillna(0).values
y = torch.tensor(card_fraud, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)
print(f"Nodes: {data.num_nodes}, Edges: {data.num_edges}")

torch.save(data, r'C:\Users\varun\Desktop\fraud-gnn\data\graph_data.pt')
print("✅ Small graph saved!")

Nodes: 2376, Edges: 10000
✅ Small graph saved!
